# 02 — Cleaning Pipeline
**Owner: Parul** | Santander Customer Satisfaction

This notebook covers all data cleaning steps:
- Load train and test
- Drop constant features
- Drop duplicate features
- Drop sparse features (>99% zeros)
- Drop highly correlated features (|corr| > 0.98)
- Fix sentinel values in var3, var36, var38

## Step 1 — Load Data

In [13]:
import pandas as pd
import numpy as np

# load train and test data
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

# check shapes
print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (76020, 371)
Test shape : (75818, 370)


## Step 2 — Save TARGET and Concat Train + Test

In [14]:
# save target separately before concat
y = train['TARGET']

# drop TARGET and ID from both before concat
train_clean = train.drop(columns=['TARGET', 'ID'])
test_clean  = test.drop(columns=['ID'])

# concat train and test together
combined = pd.concat([train_clean, test_clean], axis=0, ignore_index=True)

print("Combined shape:", combined.shape)

## Step 3 — Save Column List BEFORE Cleaning

In [15]:
# save column list before any cleaning
cols_before = combined.columns.tolist()
print("Features BEFORE cleaning:", len(cols_before))

Combined shape: (151838, 369)


## Step 4 — Drop Constant Features

In [23]:
# find columns where number of unique values is exactly 1
constant_cols = [col for col in combined.columns if combined[col].nunique() == 1]

print("Constant features found:", len(constant_cols))

# drop them
combined = combined.drop(columns=constant_cols)
print("Features remaining:", combined.shape[1])

Constant features found: 0
Features remaining: 116


## Step 5 — Drop Duplicate Features

In [28]:
# find and drop duplicate columns (transpose trick)
duplicate_cols = combined.T[combined.T.duplicated()].index.tolist()

print("Duplicate features found:", len(duplicate_cols))

# drop them
combined = combined.drop(columns=duplicate_cols)
print("Features remaining:", combined.shape[1])

Duplicate features found: 0
Features remaining: 116


## Step 6 — Drop Sparse Features (>99% zeros)

In [18]:
# find columns where more than 99% of values are zero
sparse_cols = [col for col in combined.columns if (combined[col] == 0).mean() > 0.99]

print("Sparse features found:", len(sparse_cols))

# drop them
combined = combined.drop(columns=sparse_cols)
print("Features remaining:", combined.shape[1])

Sparse features found: 183
Features remaining: 152


## Step 7 — Drop Highly Correlated Features (|corr| > 0.98)

In [19]:
# compute correlation matrix on train portion only
train_combined = combined.iloc[:len(train)]

corr_matrix = train_combined.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.98)]

print("Highly correlated features found:", len(high_corr_cols))

# drop them
combined = combined.drop(columns=high_corr_cols)
print("Features remaining:", combined.shape[1])

Highly correlated features found: 39
Features remaining: 113


## Step 8 — Fix Sentinel Values

In [20]:
# var3: -999999 means missing nationality → replace with mode + flag
combined['var3_missing'] = (combined['var3'] == -999999).astype(int)
combined.loc[combined['var3'] == -999999, 'var3'] = combined['var3'].mode()[0]
print("var3_missing flagged:", combined['var3_missing'].sum())

# var36: 99 means unknown account type → flag it
combined['var36_99_flag'] = (combined['var36'] == 99).astype(int)
print("var36_99_flag flagged:", combined['var36_99_flag'].sum())

# var38: 117310.979 is a sentinel for missing income → flag + log transform
combined['var38_flag'] = (combined['var38'] == 117310.979016494).astype(int)
combined['var38'] = np.log1p(combined['var38'])
print("var38_flag flagged:", combined['var38_flag'].sum())

var3_missing flagged: 236
var36_99_flag flagged: 60159
var38_flag flagged: 29673


## Step 9 — Split Back into Train and Test

In [21]:
# split combined back into train and test
X_train = combined.iloc[:len(train)].reset_index(drop=True)
X_test  = combined.iloc[len(train):].reset_index(drop=True)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (76020, 116)
X_test shape : (75818, 116)


## Step 10 — Cleaning Summary

In [29]:
cols_after = combined.columns.tolist()

print("===== CLEANING SUMMARY =====")
print("Features BEFORE cleaning    :", len(cols_before))
print("Constant features removed   :", len(constant_cols))
print("Duplicate features removed  :", len(duplicate_cols))
print("Sparse features removed     :", len(sparse_cols))
print("Correlated features removed :", len(high_corr_cols))
print("Sentinel flags added        : 3 (var3_missing, var36_99_flag, var38_flag)")
print("Features AFTER cleaning     :", len(cols_after))

===== CLEANING SUMMARY =====
Features BEFORE cleaning    : 116
Constant features removed   : 0
Duplicate features removed  : 0
Sparse features removed     : 183
Correlated features removed : 39
Sentinel flags added        : 3 (var3_missing, var36_99_flag, var38_flag)
Features AFTER cleaning     : 116


In [30]:
# add TARGET back to clean_train
X_train['TARGET'] = y.values

# save to processed data folder
X_train.to_csv('/Users/parulchaudhary/Desktop/SEM 2/Predictive/santander-customer-satisfaction/data/processed/clean_train.csv', index=False)
X_test.to_csv('/Users/parulchaudhary/Desktop/SEM 2/Predictive/santander-customer-satisfaction/data/processed/clean_test.csv', index=False)

print("Saved clean_train.csv:", X_train.shape)
print("Saved clean_test.csv :", X_test.shape)

Saved clean_train.csv: (76020, 117)
Saved clean_test.csv : (75818, 116)
